# OLA Ride Insights

1. Install required libraries:

In [4]:
# Install required libraries
!pip install openpyxl -q
!pip install gdown -q
!gdown --id 1fr_Q_LdN1ahZPxJxHPCyQ8YWb-LVVQsN -O OLA_DataSet.xlsx


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1fr_Q_LdN1ahZPxJxHPCyQ8YWb-LVVQsN
To: /content/OLA_DataSet.xlsx
100% 11.8M/11.8M [00:00<00:00, 65.2MB/s]


2. Data Cleaning:

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Read the dataset file
df = pd.read_excel('OLA_DataSet.xlsx', sheet_name='July')

# Check the structure of the table
# Drop the trailing empty column and Vehicle Images
df = df.loc[:, df.columns.notna()]

if 'Vehicle Images' in df.columns:
    df.drop(columns=['Vehicle Images'], inplace=True)

# Fix Date column
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Replace string 'null' with actual NaN everywhere
df.replace('null', np.nan, inplace=True)

# Fix numeric columns
for col in ['V_TAT', 'C_TAT', 'Booking_Value', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Strip whitespace from text columns
text_cols = ['Booking_Status', 'Vehicle_Type', 'Payment_Method', 'Pickup_Location',
             'Drop_Location', 'Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver',
             'Incomplete_Rides', 'Incomplete_Rides_Reason']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col].replace('nan', np.nan, inplace=True)


In [7]:
# Add helper columns
df['Hour']      = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.day_name()
df['DateOnly']  = df['Date'].dt.date


print(f'Clean shape: {df.shape}')
print(f"\nBooking statuses : {df['Booking_Status'].unique()}")
print(f"Vehicle types    : {df['Vehicle_Type'].unique()}")
print(f"Payment methods  : {df['Payment_Method'].dropna().unique()}")
df.head(3)

Clean shape: (103024, 22)

Booking statuses : ['Canceled by Driver' 'Success' 'Canceled by Customer' 'Driver Not Found']
Vehicle types    : ['Prime Sedan' 'Bike' 'Prime SUV' 'eBike' 'Mini' 'Prime Plus' 'Auto']
Payment methods  : ['Cash' 'UPI' 'Credit Card' 'Debit Card']


,Date,Time,Booking_ID,Booking_Status,Customer_ID,Vehicle_Type,Pickup_Location,Drop_Location,V_TAT,C_TAT,...,Incomplete_Rides,Incomplete_Rides_Reason,Booking_Value,Payment_Method,Ride_Distance,Driver_Ratings,Customer_Rating,Hour,DayOfWeek,DateOnly
0,2024-07-26 14:00:00,14:00:00,CNR7153255142,Canceled by Driver,CID713523,Prime Sedan,Tumkur Road,RT Nagar,NaN,NaN,...,NaN,NaN,444,NaN,0,NaN,NaN,14,Friday,2024-07-26
1,2024-07-25 22:20:00,22:20:00,CNR2940424040,Success,CID225428,Bike,Magadi Road,Varthur,203.0,30.0,...,No,NaN,158,Cash,13,4.1,4.0,22,Thursday,2024-07-25
2,2024-07-30 19:59:00,19:59:00,CNR2982357879,Success,CID270156,Prime SUV,Sahakar Nagar,Varthur,238.0,130.0,...,No,NaN,386,UPI,40,4.2,4.8,19,Tuesday,2024-07-30


In [8]:
from google.colab import files

# Download cleaned CSV (optional but useful for Power BI)
df.to_csv('ola_clean.csv', index=False)
files.download('ola_clean.csv')
print('ola_clean.csv downloaded — use this file in Power BI')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ola_clean.csv downloaded — use this file in Power BI


3. Load into SQLite (in-memory database):

In [9]:
# Create an in-memory SQLite database and load the clean data into it
conn = sqlite3.connect(':memory:')
df.to_sql('rides', conn, if_exists='replace', index=False)

# Helper function — run any SQL and display results as a table
def run_sql(query):
    result = pd.read_sql(query, conn)
    print(f'  Rows returned: {len(result)}')
    return result

print('Database ready! Table: rides')

Database ready! Table: rides


---
4. SQL Queries (All 10):
---

In [10]:
# ── Q1. Retrieve all successful bookings ─────────────────────────────────────
print('Q1 — All Successful Bookings')
run_sql("""
    SELECT *
    FROM rides
    WHERE Booking_Status = 'Success'
    LIMIT 20
""")

Q1 — All Successful Bookings
  Rows returned: 20


,Date,Time,Booking_ID,Booking_Status,Customer_ID,Vehicle_Type,Pickup_Location,Drop_Location,V_TAT,C_TAT,...,Incomplete_Rides,Incomplete_Rides_Reason,Booking_Value,Payment_Method,Ride_Distance,Driver_Ratings,Customer_Rating,Hour,DayOfWeek,DateOnly
0,2024-07-25 22:20:00,22:20:00.000000,CNR2940424040,Success,CID225428,Bike,Magadi Road,Varthur,203.0,30.0,...,No,None,158,Cash,13,4.1,4.0,22,Thursday,2024-07-25
1,2024-07-30 19:59:00,19:59:00.000000,CNR2982357879,Success,CID270156,Prime SUV,Sahakar Nagar,Varthur,238.0,130.0,...,No,None,386,UPI,40,4.2,4.8,19,Tuesday,2024-07-30
2,2024-07-02 09:02:00,09:02:00.000000,CNR1797421769,Success,CID939555,Mini,Rajajinagar,Chamarajpet,252.0,80.0,...,No,None,822,Credit Card,45,4.0,3.0,9,Tuesday,2024-07-02
3,2024-07-13 04:42:00,04:42:00.000000,CNR8787177882,Success,CID802429,Mini,Kadugodi,Vijayanagar,231.0,90.0,...,No,None,173,UPI,41,3.4,4.6,4,Saturday,2024-07-13
4,2024-07-23 09:51:00,09:51:00.000000,CNR3612067560,Success,CID476071,Bike,Tumkur Road,Whitefield,133.0,40.0,...,No,None,140,Cash,49,3.2,4.5,9,Tuesday,2024-07-23
5,2024-07-29 23:33:00,23:33:00.000000,CNR4787583516,Success,CID923404,Prime Plus,Hosur Road,Jayanagar,35.0,55.0,...,No,None,164,Cash,46,4.5,3.4,23,Monday,2024-07-29
6,2024-07-26 04:03:00,04:03:00.000000,CNR7943634301,Success,CID647026,Prime Plus,Kammanahalli,Rajajinagar,238.0,95.0,...,No,None,399,Cash,18,3.9,4.4,4,Friday,2024-07-26
7,2024-07-27 13:18:00,13:18:00.000000,CNR4524472111,Success,CID540929,Auto,Cox Town,Yelahanka,126.0,35.0,...,No,None,330,UPI,8,3.0,4.8,13,Saturday,2024-07-27
8,2024-07-16 09:54:00,09:54:00.000000,CNR8181602032,Success,CID167642,Bike,Indiranagar,MG Road,70.0,95.0,...,No,None,378,UPI,18,4.8,4.1,9,Tuesday,2024-07-16
9,2024-07-02 10:25:00,10:25:00.000000,CNR8090918544,Success,CID640151,Bike,Magadi Road,HSR Layout,126.0,95.0,...,No,None,343,UPI,23,3.7,3.6,10,Tuesday,2024-07-02


In [11]:
# ── Q2. Average ride distance for each vehicle type ───────────────────────────
print('Q2 — Average Ride Distance per Vehicle Type')
run_sql("""
    SELECT
        Vehicle_Type,
        ROUND(AVG(Ride_Distance), 2) AS Avg_Ride_Distance_km
    FROM rides
    GROUP BY Vehicle_Type
    ORDER BY Avg_Ride_Distance_km DESC
""")

Q2 — Average Ride Distance per Vehicle Type
  Rows returned: 7


,Vehicle_Type,Avg_Ride_Distance_km
0,Prime Sedan,15.76
1,eBike,15.58
2,Bike,15.53
3,Mini,15.51
4,Prime Plus,15.45
5,Prime SUV,15.27
6,Auto,6.24


In [12]:
# ── Q3. Total number of cancelled rides by customers ─────────────────────────
print('Q3 — Total Customer Cancellations')
run_sql("""
    SELECT COUNT(*) AS Total_Customer_Cancellations
    FROM rides
    WHERE Booking_Status = 'Canceled by Customer'
""")

Q3 — Total Customer Cancellations
  Rows returned: 1


,Total_Customer_Cancellations
0,10499


In [13]:
# ── Q4. Top 5 customers by number of rides booked ────────────────────────────
print('Q4 — Top 5 Customers by Ride Count')
run_sql("""
    SELECT
        Customer_ID,
        COUNT(*) AS Total_Rides
    FROM rides
    GROUP BY Customer_ID
    ORDER BY Total_Rides DESC
    LIMIT 5
""")

Q4 — Top 5 Customers by Ride Count
  Rows returned: 5


,Customer_ID,Total_Rides
0,CID954071,5
1,CID980727,4
2,CID969725,4
3,CID966929,4
4,CID952434,4


In [14]:
# ── Q5. Driver cancellations due to personal/car-related issues ───────────────
print('Q5 — Driver Cancellations: Personal & Car Related Issues')
run_sql("""
    SELECT COUNT(*) AS Driver_Cancel_Personal_Car
    FROM rides
    WHERE Canceled_Rides_by_Driver = 'Personal & Car related issue'
""")

Q5 — Driver Cancellations: Personal & Car Related Issues
  Rows returned: 1


,Driver_Cancel_Personal_Car
0,6542


In [15]:
# ── Q6. Max & min driver ratings for Prime Sedan ─────────────────────────────
print('Q6 — Max & Min Driver Ratings for Prime Sedan')
run_sql("""
    SELECT
        MAX(Driver_Ratings) AS Max_Driver_Rating,
        MIN(Driver_Ratings) AS Min_Driver_Rating
    FROM rides
    WHERE Vehicle_Type = 'Prime Sedan'
      AND Driver_Ratings IS NOT NULL
""")

Q6 — Max & Min Driver Ratings for Prime Sedan
  Rows returned: 1


,Max_Driver_Rating,Min_Driver_Rating
0,5.0,3.0


In [16]:
# ── Q7. All rides paid via UPI ────────────────────────────────────────────────
print('Q7 — Rides Paid via UPI')
run_sql("""
    SELECT *
    FROM rides
    WHERE Payment_Method = 'UPI'
    LIMIT 20
""")

Q7 — Rides Paid via UPI
  Rows returned: 20


,Date,Time,Booking_ID,Booking_Status,Customer_ID,Vehicle_Type,Pickup_Location,Drop_Location,V_TAT,C_TAT,...,Incomplete_Rides,Incomplete_Rides_Reason,Booking_Value,Payment_Method,Ride_Distance,Driver_Ratings,Customer_Rating,Hour,DayOfWeek,DateOnly
0,2024-07-30 19:59:00,19:59:00.000000,CNR2982357879,Success,CID270156,Prime SUV,Sahakar Nagar,Varthur,238.0,130.0,...,No,None,386,UPI,40,4.2,4.8,19,Tuesday,2024-07-30
1,2024-07-13 04:42:00,04:42:00.000000,CNR8787177882,Success,CID802429,Mini,Kadugodi,Vijayanagar,231.0,90.0,...,No,None,173,UPI,41,3.4,4.6,4,Saturday,2024-07-13
2,2024-07-27 13:18:00,13:18:00.000000,CNR4524472111,Success,CID540929,Auto,Cox Town,Yelahanka,126.0,35.0,...,No,None,330,UPI,8,3.0,4.8,13,Saturday,2024-07-27
3,2024-07-16 09:54:00,09:54:00.000000,CNR8181602032,Success,CID167642,Bike,Indiranagar,MG Road,70.0,95.0,...,No,None,378,UPI,18,4.8,4.1,9,Tuesday,2024-07-16
4,2024-07-02 10:25:00,10:25:00.000000,CNR8090918544,Success,CID640151,Bike,Magadi Road,HSR Layout,126.0,95.0,...,No,None,343,UPI,23,3.7,3.6,10,Tuesday,2024-07-02
5,2024-07-09 11:11:00,11:11:00.000000,CNR9975925287,Success,CID162055,Prime SUV,Magadi Road,RT Nagar,42.0,30.0,...,No,None,343,UPI,17,3.0,3.8,11,Tuesday,2024-07-09
6,2024-07-19 21:18:00,21:18:00.000000,CNR4443921904,Success,CID654618,Mini,Tumkur Road,Koramangala,231.0,50.0,...,No,None,286,UPI,44,4.0,3.3,21,Friday,2024-07-19
7,2024-07-25 03:44:00,03:44:00.000000,CNR7194303296,Success,CID538245,Mini,Mysore Road,Hennur,175.0,50.0,...,No,None,141,UPI,35,4.7,3.1,3,Thursday,2024-07-25
8,2024-07-15 17:11:00,17:11:00.000000,CNR6494005067,Success,CID805360,Auto,Yelahanka,Malleshwaram,84.0,60.0,...,No,None,214,UPI,2,3.3,4.5,17,Monday,2024-07-15
9,2024-07-14 05:25:00,05:25:00.000000,CNR7142279862,Success,CID378034,eBike,Yeshwanthpur,JP Nagar,210.0,45.0,...,No,None,461,UPI,49,4.5,3.1,5,Sunday,2024-07-14


In [17]:
# ── Q8. Average customer rating per vehicle type ──────────────────────────────
print('Q8 — Average Customer Rating per Vehicle Type')
run_sql("""
    SELECT
        Vehicle_Type,
        ROUND(AVG(Customer_Rating), 2) AS Avg_Customer_Rating
    FROM rides
    WHERE Customer_Rating IS NOT NULL
    GROUP BY Vehicle_Type
    ORDER BY Avg_Customer_Rating DESC
""")

Q8 — Average Customer Rating per Vehicle Type
  Rows returned: 7


,Vehicle_Type,Avg_Customer_Rating
0,Prime Plus,4.01
1,Prime Sedan,4.00
2,Prime SUV,4.00
3,Mini,4.00
4,Auto,4.00
5,eBike,3.99
6,Bike,3.99


In [18]:
# ── Q9. Total booking value of successfully completed rides ───────────────────
print('Q9 — Total Revenue from Successful Rides')
run_sql("""
    SELECT
        ROUND(SUM(Booking_Value), 2) AS Total_Successful_Booking_Value
    FROM rides
    WHERE Booking_Status = 'Success'
""")

Q9 — Total Revenue from Successful Rides
  Rows returned: 1


,Total_Successful_Booking_Value
0,35080467.0


In [19]:
# ── Q10. All incomplete rides with reason ─────────────────────────────────────
print('Q10 — Incomplete Rides with Reason')
run_sql("""
    SELECT
        Booking_ID,
        Customer_ID,
        Vehicle_Type,
        Pickup_Location,
        Drop_Location,
        Booking_Value,
        Incomplete_Rides_Reason
    FROM rides
    WHERE Incomplete_Rides = 'Yes'
""")

Q10 — Incomplete Rides with Reason
  Rows returned: 3926


,Booking_ID,Customer_ID,Vehicle_Type,Pickup_Location,Drop_Location,Booking_Value,Incomplete_Rides_Reason
0,CNR5176704322,CID296026,Prime Plus,KR Puram,Hennur,1102,Customer Demand
1,CNR9312632867,CID649563,Prime SUV,Tumkur Road,Mysore Road,2936,Vehicle Breakdown
2,CNR7924302885,CID517661,Auto,Magadi Road,Nagarbhavi,685,Customer Demand
3,CNR1640228587,CID190281,Prime Sedan,Kengeri,Mysore Road,704,Other Issue
4,CNR7623690602,CID526261,Bike,Cox Town,Malleshwaram,558,Other Issue
...,...,...,...,...,...,...,...
3921,CNR5546265534,CID118412,Prime SUV,Bellandur,Bellandur,297,Customer Demand
3922,CNR1954831907,CID771129,Auto,Magadi Road,Hennur,474,Vehicle Breakdown
3923,CNR1271821250,CID112738,Auto,Marathahalli,Peenya,306,Vehicle Breakdown
3924,CNR4652634649,CID382466,Mini,Koramangala,Devanahalli,315,Vehicle Breakdown
